# CRUD Operations

## Overview
DynamoDB provides four single-item operations and several batch/transaction operations. Unlike SQL's flexible `UPDATE ... WHERE`, every DynamoDB write targets a specific item by its complete primary key.

**Single-item operations:**

| Operation | SQL equivalent | Notes |
|---|---|---|
| `PutItem` | `INSERT ... ON CONFLICT REPLACE` | Replaces entire item if key exists |
| `GetItem` | `SELECT ... WHERE pk = ? AND sk = ?` | Exact key lookup only; no range queries |
| `UpdateItem` | `UPDATE ... SET col = val WHERE pk = ?` | Modify specific attributes; key is required |
| `DeleteItem` | `DELETE ... WHERE pk = ? AND sk = ?` | Requires full primary key |

**Key principles:**
- Every write and read requires the full primary key (PK + SK if composite)
- `UpdateItem` uses an `UpdateExpression` language (`SET`, `REMOVE`, `ADD`, `DELETE`)
- `ConditionExpression` makes any write conditional — preventing overwrites, enforcing constraints
- DynamoDB has hundreds of **reserved words** that must be aliased with `ExpressionAttributeNames`

**Local setup:**
```bash
docker run -p 8000:8000 amazon/dynamodb-local
pip install boto3
```

---

In [1]:

print("=== DynamoDB setup with boto3 ===")
setup_code = '''
import boto3
from boto3.dynamodb.conditions import Key, Attr
from decimal import Decimal
import json

# ── Connect to DynamoDB Local ────────────────────────────────────
dynamodb = boto3.resource(
    "dynamodb",
    region_name="ca-central-1",
    endpoint_url="http://localhost:8000",   # local only; remove for AWS
    aws_access_key_id="local",              # any value for local
    aws_secret_access_key="local",          # any value for local
)

# ── Create the ecological observations table ─────────────────────
table = dynamodb.create_table(
    TableName="EcoObservations",
    KeySchema=[
        {"AttributeName": "site_id",  "KeyType": "HASH"},   # Partition key
        {"AttributeName": "obs_sk",   "KeyType": "RANGE"},  # Sort key
    ],
    AttributeDefinitions=[
        {"AttributeName": "site_id",  "AttributeType": "S"},
        {"AttributeName": "obs_sk",   "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",  # On-Demand
)
table.wait_until_exists()
print("Table created:", table.table_name, "| Status:", table.table_status)

# Sort key (obs_sk) encodes entity type + date + id:
# "OBS#2023-04-10#001"  → observation on April 10 with id 001
# "WQ#2023-04-10"       → water quality sample on April 10
# "META#SITE"           → site metadata record
'''
print(setup_code)


=== DynamoDB setup with boto3 ===

import boto3
from boto3.dynamodb.conditions import Key, Attr
from decimal import Decimal
import json

# ── Connect to DynamoDB Local ────────────────────────────────────
dynamodb = boto3.resource(
    "dynamodb",
    region_name="ca-central-1",
    endpoint_url="http://localhost:8000",   # local only; remove for AWS
    aws_access_key_id="local",              # any value for local
    aws_secret_access_key="local",          # any value for local
)

# ── Create the ecological observations table ─────────────────────
table = dynamodb.create_table(
    TableName="EcoObservations",
    KeySchema=[
        {"AttributeName": "site_id",  "KeyType": "HASH"},   # Partition key
        {"AttributeName": "obs_sk",   "KeyType": "RANGE"},  # Sort key
    ],
    AttributeDefinitions=[
        {"AttributeName": "site_id",  "AttributeType": "S"},
        {"AttributeName": "obs_sk",   "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",  # On-Demand
)
tabl

---
## PutItem: inserting and conditional writes

In [2]:

print("=== PutItem: insert or replace a full item ===")
put_code = '''
table = dynamodb.Table("EcoObservations")

# ── Put a site metadata item ─────────────────────────────────────
table.put_item(Item={
    "site_id":   "SITE#001",
    "obs_sk":    "META#SITE",           # sort key for metadata
    "site_name": "Fundy Estuary",
    "region":    "Atlantic",
    "latitude":  Decimal("45.78"),      # DynamoDB requires Decimal not float
    "longitude": Decimal("-64.52"),
    "active":    True,
    "tags":      {"wetland", "estuary"} # SS (string set)
})

# ── Put an observation item ──────────────────────────────────────
table.put_item(Item={
    "site_id":    "SITE#001",
    "obs_sk":     "OBS#2023-04-10#001",
    "species_id": "SP#SALMON",
    "common_name":"Atlantic Salmon",
    "count":      12,
    "life_stage": "Adult",
    "method":     "Electrofishing",
    "crew_id":    "CREW#001",
    "at_risk":    True,
})

# ── PutItem with condition: only insert if item does not exist ───
from botocore.exceptions import ClientError
try:
    table.put_item(
        Item={"site_id": "SITE#001", "obs_sk": "META#SITE", "site_name": "Duplicate"},
        ConditionExpression="attribute_not_exists(site_id)"
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "ConditionalCheckFailedException":
        print("Item already exists — not overwritten")
'''
print(put_code)


=== PutItem: insert or replace a full item ===

table = dynamodb.Table("EcoObservations")

# ── Put a site metadata item ─────────────────────────────────────
table.put_item(Item={
    "site_id":   "SITE#001",
    "obs_sk":    "META#SITE",           # sort key for metadata
    "site_name": "Fundy Estuary",
    "region":    "Atlantic",
    "latitude":  Decimal("45.78"),      # DynamoDB requires Decimal not float
    "longitude": Decimal("-64.52"),
    "active":    True,
    "tags":      {"wetland", "estuary"} # SS (string set)
})

# ── Put an observation item ──────────────────────────────────────
table.put_item(Item={
    "site_id":    "SITE#001",
    "obs_sk":     "OBS#2023-04-10#001",
    "species_id": "SP#SALMON",
    "common_name":"Atlantic Salmon",
    "count":      12,
    "life_stage": "Adult",
    "method":     "Electrofishing",
    "crew_id":    "CREW#001",
    "at_risk":    True,
})

# ── PutItem with condition: only insert if item does not exist ───
from botocore.exceptions 

---
## GetItem and UpdateItem

In [3]:

print("=== GetItem and UpdateItem ===")
get_update_code = '''
# ── GetItem: fetch exactly one item by full primary key ──────────
response = table.get_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    ProjectionExpression="site_id, common_name, #cnt, life_stage",
    ExpressionAttributeNames={"#cnt": "count"},  # 'count' is a reserved word
)
item = response.get("Item")
print(item)

# ── UpdateItem: change specific attributes ───────────────────────
# SQL equivalent: UPDATE observations SET count = 15 WHERE site_id='SITE#001'...
table.update_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    UpdateExpression="SET #cnt = :new_count, verified = :v",
    ExpressionAttributeNames={"#cnt": "count"},
    ExpressionAttributeValues={":new_count": 15, ":v": True},
)

# ── UpdateItem: atomic increment ────────────────────────────────
table.update_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    UpdateExpression="ADD #cnt :delta",
    ExpressionAttributeNames={"#cnt": "count"},
    ExpressionAttributeValues={":delta": 3},
)

# ── UpdateItem: conditional update ──────────────────────────────
# Only update if current count is less than 20
table.update_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    UpdateExpression="SET #cnt = :new_count",
    ConditionExpression="#cnt < :max_count",
    ExpressionAttributeNames={"#cnt": "count"},
    ExpressionAttributeValues={":new_count": 18, ":max_count": 20},
)
'''
print(get_update_code)


=== GetItem and UpdateItem ===

# ── GetItem: fetch exactly one item by full primary key ──────────
response = table.get_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    ProjectionExpression="site_id, common_name, #cnt, life_stage",
    ExpressionAttributeNames={"#cnt": "count"},  # 'count' is a reserved word
)
item = response.get("Item")
print(item)

# ── UpdateItem: change specific attributes ───────────────────────
# SQL equivalent: UPDATE observations SET count = 15 WHERE site_id='SITE#001'...
table.update_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    UpdateExpression="SET #cnt = :new_count, verified = :v",
    ExpressionAttributeNames={"#cnt": "count"},
    ExpressionAttributeValues={":new_count": 15, ":v": True},
)

# ── UpdateItem: atomic increment ────────────────────────────────
table.update_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    UpdateExpression="ADD #cnt :delta",
    ExpressionAttribut

---
## DeleteItem, BatchWrite, and Transactions

In [4]:

print("=== DeleteItem and batch operations ===")
batch_code = '''
# ── DeleteItem ───────────────────────────────────────────────────
table.delete_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    ConditionExpression="attribute_exists(site_id)"  # safety guard
)

# ── BatchWriteItem: up to 25 puts/deletes in one request ─────────
# BatchWrite does NOT support UpdateItem
with table.batch_writer() as batch:
    for obs in observations_list:         # list of dicts
        batch.put_item(Item=obs)
    batch.delete_item(
        Key={"site_id": "SITE#OLD", "obs_sk": "OBS#2022-01-01#001"}
    )
# batch_writer handles chunking into 25-item batches automatically
# and retries UnprocessedItems on ProvisionedThroughputExceeded

# ── BatchGetItem: up to 100 gets in one request ──────────────────
response = dynamodb.batch_get_item(
    RequestItems={
        "EcoObservations": {
            "Keys": [
                {"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
                {"site_id": "SITE#002", "obs_sk": "OBS#2023-05-01#003"},
                {"site_id": "SITE#003", "obs_sk": "META#SITE"},
            ]
        }
    }
)
items = response["Responses"]["EcoObservations"]

# ── TransactWriteItems: all-or-nothing across multiple items ─────
dynamodb.meta.client.transact_write_items(
    TransactItems=[
        {"Put": {"TableName": "EcoObservations", "Item": {
            "site_id": "SITE#001", "obs_sk": "OBS#2023-10-01#099",
            "species_id": "SP#TURTLE", "count": 1
        }}},
        {"Update": {"TableName": "EcoObservations",
            "Key": {"site_id": "SITE#001", "obs_sk": "META#SITE"},
            "UpdateExpression": "ADD obs_count :one",
            "ExpressionAttributeValues": {":one": 1},
        }},
    ]
)
# Both succeed or both fail — ACID across items (up to 100 items / 4 MB)
'''
print(batch_code)

print("Operation comparison:")
ops = [
    ("PutItem",          "Insert or replace a full item (no partial update)"),
    ("GetItem",          "Fetch one item by exact primary key"),
    ("UpdateItem",       "Modify specific attributes without replacing the item"),
    ("DeleteItem",       "Remove one item by primary key"),
    ("BatchWriteItem",   "Up to 25 puts/deletes; no UpdateItem; no transactions"),
    ("BatchGetItem",     "Up to 100 gets across one or more tables"),
    ("TransactWriteItems","Up to 100 all-or-nothing writes (ACID)"),
    ("TransactGetItems", "Up to 100 strongly consistent reads (ACID)"),
]
for op, desc in ops:
    print(f"  {op:<24s}  {desc}")


=== DeleteItem and batch operations ===

# ── DeleteItem ───────────────────────────────────────────────────
table.delete_item(
    Key={"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
    ConditionExpression="attribute_exists(site_id)"  # safety guard
)

# ── BatchWriteItem: up to 25 puts/deletes in one request ─────────
# BatchWrite does NOT support UpdateItem
with table.batch_writer() as batch:
    for obs in observations_list:         # list of dicts
        batch.put_item(Item=obs)
    batch.delete_item(
        Key={"site_id": "SITE#OLD", "obs_sk": "OBS#2022-01-01#001"}
    )
# batch_writer handles chunking into 25-item batches automatically
# and retries UnprocessedItems on ProvisionedThroughputExceeded

# ── BatchGetItem: up to 100 gets in one request ──────────────────
response = dynamodb.batch_get_item(
    RequestItems={
        "EcoObservations": {
            "Keys": [
                {"site_id": "SITE#001", "obs_sk": "OBS#2023-04-10#001"},
                {"site_i

---
## Common Pitfalls

**1. Using floats instead of Decimal for numeric attributes**
boto3 raises a `TypeError` if you pass a Python `float` as an attribute value. DynamoDB stores numbers as strings internally and requires `Decimal` in boto3 to prevent floating-point precision loss. Always use `from decimal import Decimal` and convert: `Decimal('7.25')` not `7.25`.

**2. PutItem silently overwrites existing items**
`put_item()` replaces the entire item if the key already exists. To prevent accidental overwrites, always add `ConditionExpression='attribute_not_exists(site_id)'` when inserting new items. Use `update_item()` with specific `UpdateExpression` when you only want to change some attributes.

**3. Reserved words in attribute names causing expression errors**
DynamoDB has hundreds of reserved words (`count`, `name`, `status`, `type`, `value`, `date`, etc.). Using them directly in expressions raises a `ValidationException`. Always use `ExpressionAttributeNames={'#cnt': 'count'}` to alias reserved words, and `#cnt` in the expression.

**4. Not handling UnprocessedItems in BatchWriteItem**
`batch_write_item()` can return `UnprocessedItems` when throughput is exceeded. If you ignore this field, those items are silently lost. The `batch_writer()` context manager handles retries automatically -- prefer it over manual `batch_write_item()` calls.

**5. TransactWriteItems across different AWS regions or accounts**
DynamoDB transactions only work within a single AWS region and account. Cross-region or cross-account consistency requires an application-level saga pattern, not DynamoDB transactions.

**6. UpdateItem with SET creating new attributes unintentionally**
`SET new_field = :val` creates `new_field` if it does not exist. If you only want to update an existing field, add a condition: `ConditionExpression='attribute_exists(new_field)'`. Without the guard, typos in attribute names silently create orphaned fields.


---
*sql_methods_library - Samantha McGarrigle*